# N02 - Descriptives, missingness, survey design, associations (plan stage 2)
Gate: primary predictor set frozen; unresolved variables explicitly excluded.

**Leakage rule for this notebook:** it never builds outcome-by-predictor tables and never uses the outcome to choose predictors. The only outcome number shown is overall prevalence (T2). Predictor choice is made from audit findings and the plan only.

In [ ]:
# --- setup (same in every notebook) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab - using local folder.")
import os, sys, time
import numpy as np
import pandas as pd
BASE_DIR = os.environ.get("THESIS_BASE_DIR", "/content/drive/MyDrive/THESIS_V2")
sys.path.append(BASE_DIR + "/code")
import bdhs_helpers as h
h.make_all_folders()
h.set_all_seeds(h.SEED_SPLIT)
h.print_environment("N02")

In [ ]:
h.require_signoff("n01_signoff.txt", "finish N01 (or N01B) and set SIGN_OFF = True there")
cohort = pd.read_csv(h.PRIVATE_DIR + "/cohort_private.csv")
print("cohort:", cohort.shape)

### Special-code decisions (written down, not silent)
Survey special codes are **not** ordinary categories. Each one needs a decision: keep as its own named level, treat as missing, or (bigger change, needs a plan update) drop rows / drop the variable.

The table below is my **proposed default**, based on what the data shows (counts are printed under it). It is your decision - edit `decisions` if you disagree. Notes:
- **v113 / v116 code 97** sits on exactly the same children in both variables. That pattern looks like a questionnaire-universe code, not a water or toilet type. It is kept as a separately named level `not_applicable_97` so it is never read as a housing category. I cannot confirm its meaning from this file (no value labels are stored in it, and `v135` is not in the extract) - check the DHS recode manual.
- **v701 code 8 and v705 code 98** are "don't know" (7 children each) and are treated as missing.
- The 52 children with missing v701/v705 may be structural (no partner) or item non-response. `v501` is not in the extract, so this cannot be settled here (U2 stays open); they stay as `missing`.
- Alternatives that need a plan change first: exclude the 406 children, or drop v113/v116.

In [ ]:
decisions = h.default_coding_decisions()      # <- edit this table if you decide differently
print(decisions.to_string(index=False))
print()
for i in range(len(decisions)):
    var = decisions["variable"].iloc[i]
    code = float(decisions["code"].iloc[i])
    print(var, "code", decisions["code"].iloc[i], "-> action:", decisions["action"].iloc[i], "| children affected:", int((cohort[var] == code).sum()))
h.save_table(decisions, h.OUTPUT_DIR + "/coding_decisions.csv")
clean = h.clean_predictors(cohort, h.PREDICTORS_DEFAULT, decisions)

### Survey design: PSUs, strata, sparse strata

In [ ]:
print("PSU variable v021 unique:", cohort["v021"].nunique(), "| stratum v022 unique:", cohort["v022"].nunique())
print("v001 == v021 everywhere:", bool((cohort["v001"] == cohort["v021"]).all()),
      "| v022 == v023 everywhere:", bool((cohort["v022"] == cohort["v023"]).all()))

rows_per_psu = cohort.groupby("v021").size()
print("rows per PSU: min", rows_per_psu.min(), "| median", rows_per_psu.median(), "| max", rows_per_psu.max())

psu_per_stratum = cohort.groupby("v022")["v021"].nunique().reset_index()
psu_per_stratum.columns = ["stratum_v022", "n_psu"]
rows_per_stratum = cohort.groupby("v022").size().reset_index()
rows_per_stratum.columns = ["stratum_v022", "n_rows"]
strata_table = psu_per_stratum.merge(rows_per_stratum, on="stratum_v022")
sparse = strata_table[strata_table["n_psu"] < 2]
print("strata with fewer than 2 PSUs (not estimable for a stratified PSU bootstrap):", len(sparse))
if len(sparse) > 0:
    print(sparse.to_string(index=False))
h.save_table(strata_table, h.OUTPUT_DIR + "/n02_strata_psu_counts.csv")

### Missingness and special codes (T2)

In [ ]:
rows = []
for var in h.PREDICTORS_DEFAULT:
    n_missing = int(cohort[var].isna().sum())
    rows.append([var, "raw missing (NaN)", n_missing, round(100.0 * n_missing / len(cohort), 2)])
n_flag = int((cohort["v445"] >= 9990).sum())
rows.append(["v445", "special code >= 9990 (-> missing after cleaning)", n_flag, round(100.0 * n_flag / len(cohort), 2)])
for var in ["v113", "v116"]:
    n97 = int((cohort[var] == 97).sum())
    rows.append([var, "code 97 (kept as its own level)", n97, round(100.0 * n97 / len(cohort), 2)])
miss_df = pd.DataFrame(rows, columns=["variable", "what", "n", "percent"])
print(miss_df.to_string(index=False))
h.save_table(miss_df, h.OUTPUT_DIR + "/n02_missingness.csv")

In [ ]:
# category counts after cleaning (what the models will see) - outcome NOT used
cat_rows = []
for var in h.PREDICTORS_DEFAULT:
    if var in h.NUMERIC_VARS:
        continue
    counts = clean[var].value_counts()
    for level in sorted(list(counts.index)):
        rare = "yes" if counts[level] < h.MIN_CAT_COUNT else ""
        cat_rows.append([var, level, int(counts[level]), rare])
cat_df = pd.DataFrame(cat_rows, columns=["variable", "level", "n", "rare_in_full_cohort(<" + str(h.MIN_CAT_COUNT) + ")"])
print(cat_df.to_string(index=False))
h.save_table(cat_df, h.OUTPUT_DIR + "/n02_category_counts.csv")

numeric_list = []
for var in h.PREDICTORS_DEFAULT:
    if var in h.NUMERIC_VARS:
        numeric_list.append(var)
print()
print(clean[numeric_list].describe().round(2).to_string())

### Prevalence (T2)
Unweighted is the study estimand. The weighted line uses `v005/1,000,000` **for description only** - the plan says this weight is not verified for child anthropometry (U4).

In [ ]:
weight = cohort["v005"] / 1000000.0
unweighted = cohort["stunted"].mean()
weighted = (weight * cohort["stunted"]).sum() / weight.sum()
prev = pd.DataFrame([["unweighted (study cohort)", round(unweighted, 4), len(cohort)],
                     ["weighted with v005/1e6 (descriptive only, weight NOT verified - U4)", round(weighted, 4), len(cohort)]],
                    columns=["estimate", "stunting_prevalence", "n"])
print(prev.to_string(index=False))
h.save_table(prev, h.OUTPUT_DIR + "/n02_prevalence.csv")

### T3 - predictor relationships
Spearman for numeric pairs, Cramer's V for everything else (numeric variables cut into 4 quantile bins for the V check only). Flag = |value| >= 0.5. Flags are a descriptive warning, not a deletion rule.

In [ ]:
names = h.PREDICTORS_DEFAULT
binned = pd.DataFrame(index=clean.index)
for var in names:
    if var in h.NUMERIC_VARS:
        binned[var] = pd.qcut(clean[var], 4, duplicates="drop").astype(str)
    else:
        binned[var] = clean[var]

pair_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a = names[i]
        b = names[j]
        if (a in h.NUMERIC_VARS) and (b in h.NUMERIC_VARS):
            value = clean[a].corr(clean[b], method="spearman")
            measure = "spearman"
        else:
            value = h.cramers_v(binned[a], binned[b])
            measure = "cramers_v"
        flag = "FLAG" if abs(value) >= 0.5 else ""
        pair_rows.append([a, b, measure, round(float(value), 3), flag])
pairs = pd.DataFrame(pair_rows, columns=["var_1", "var_2", "measure", "value", "flag"])
pairs["abs_value"] = pairs["value"].abs()
pairs = pairs.sort_values("abs_value", ascending=False).drop(columns=["abs_value"])
print(pairs.head(15).to_string(index=False))
h.save_table(pairs, h.OUTPUT_DIR + "/T3_predictor_associations.csv")

### Freeze the primary predictor set (D2)
Defaults follow plan 3.3: the 14 candidate variables are primary; `m14`, `m15`, `v438` are excluded.
Edit `FROZEN_PREDICTORS` **only** if the N01/N02 audit says a variable cannot be coded defensibly (for example v701/v705). Record the reason. Do not use outcome results to decide.

In [ ]:
FROZEN_PREDICTORS = ["b19", "b4", "bord", "b11", "v012", "v106", "v445", "v190", "v025", "v024", "v113", "v116", "v701", "v705"]
DROP_REASON = ""   # if you removed a variable above, write the reason here

excluded_list = [
    ["m14", "excluded", "eligible-birth universe unverified; missingness age-dependent (plan 3.3)"],
    ["m15", "excluded", "eligible-birth universe unverified; missingness age-dependent (plan 3.3)"],
    ["v438", "excluded", "not in current extract; not added silently (plan 3.3, U3)"],
]
plan_set = h.PREDICTORS_DEFAULT

freeze_rows = []
for var in FROZEN_PREDICTORS:
    if var in h.FORBIDDEN_PREDICTORS:
        raise ValueError(var + " is on the forbidden-predictor list")
    freeze_rows.append([var, "primary", "plan 3.3 candidate set"])
for var in plan_set:
    if var not in FROZEN_PREDICTORS:
        freeze_rows.append([var, "excluded", "removed after audit: " + DROP_REASON])
for item in excluded_list:
    freeze_rows.append(item)
for i in range(len(decisions)):
    freeze_rows.append([decisions["variable"].iloc[i] + " code " + decisions["code"].iloc[i], "coding_decision",
                        decisions["action"].iloc[i] + " | " + decisions["reason"].iloc[i]])
freeze_df = pd.DataFrame(freeze_rows, columns=["variable", "status", "reason"])
print(freeze_df.to_string(index=False))
h.save_table(freeze_df, h.OUTPUT_DIR + "/primary_predictors.csv")

In [ ]:
SIGN_OFF = False      # True only when the predictor set above is final
if SIGN_OFF:
    f = open(h.OUTPUT_DIR + "/n02_signoff.txt", "w")
    f.write("N02 signed off. predictors = " + str(FROZEN_PREDICTORS) + "\n")
    f.close()
    print("Gate passed. N03 may run.")
else:
    print("Gate NOT passed yet. N03 will refuse to run.")